# Prompt Templates

Raw f-strings and copy-pasted prompt text do not scale. Production LLM applications need **reusable**, **versioned** prompts with **variable slots**, **role structure**, and optional **few-shot examples**. LangChain **prompt templates** are Runnables that turn input dicts into **`ChatPromptValue`** objects — formatted message lists ready for chat models.

This notebook covers **`ChatPromptTemplate`** (the standard for chat models), template variables, **`from_messages`** vs **`from_template`**, **`MessagesPlaceholder`** for dynamic history, **`.partial()`** to freeze variables, **few-shot** prompt templates, per-message template classes, Jinja2 formatting, composing prompts in LCEL chains, and RAG-oriented prompt patterns used later in **11. RAG with LCEL**.

**03. Chat Models and Messages** explained message types and model invocation. **05. Output Parsers and Structured Output** covers extracting structured data from model replies — templates control what goes in; parsers control what comes out.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core
import numpy as np

np.set_printoptions(precision=4, suppress=True)

print("langchain-core:", langchain_core.__version__)

---

## 1. Why Prompt Templates?

Without templates, prompts scatter across your codebase:

```python
# Fragile: duplicated, untested, no variable validation
prompt = f"You are an expert. Answer about {topic}. Question: {question}"
```

Problems with ad hoc strings:

| Issue | Consequence |
|-------|-------------|
| No variable validation | Missing `{topic}` at runtime |
| Mixed roles in one string | Model ignores system vs user boundaries |
| Hard to version | Cannot diff prompt changes in git cleanly |
| No reuse | Same system prompt copy-pasted in 10 files |
| Testing difficulty | Must mock entire API to test prompt shape |

**Prompt templates** separate **structure** (roles, instructions, slots) from **runtime values** (user question, retrieved context). They are **Runnables** — compose with `|` in LCEL (**02. Runnable Protocol and LCEL**).

---

## 2. ChatPromptTemplate — The Standard Template

**`ChatPromptTemplate`** produces a list of messages for **`BaseChatModel`**. It is the template class you use in nearly all LangChain 1.x chat applications.

```
dict of variables ──► ChatPromptTemplate.invoke() ──► ChatPromptValue ──► messages[]
```

Two common constructors:

| Method | Input shape | Best for |
|--------|-------------|----------|
| **`from_messages([...])`** | List of role tuples or message templates | Multi-role prompts, placeholders |
| **`from_template(str)`** | Single string → one human message | Quick one-turn prompts |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

multi_role = ChatPromptTemplate.from_messages([
    ("system", "You explain {topic} to a {audience}. Be concise."),
    ("human", "{question}"),
])

single_turn = ChatPromptTemplate.from_template(
    "Summarize the following in {max_words} words:\n\n{text}"
)

print("input_variables (multi_role):", multi_role.input_variables)
print("input_variables (single_turn):", single_turn.input_variables)

---

## 3. Template Variables

Variables use **`{name}`** syntax (Python f-string style by default). LangChain extracts **`input_variables`** automatically — the keys you must supply to `invoke()`.

Literal braces in the template require doubling: `{{` and `}}`.

In [ ]:
from langchain_core.prompts import get_template_variables

template_text = "User {name} asked about {topic}. Reply in {language}."
print("detected variables:", get_template_variables(template_text, "f-string"))

literal_braces = ChatPromptTemplate.from_template(
    "Return JSON with keys {{"id"}} and {{"name"}} for: {entity}"
)

formatted = literal_braces.invoke({"entity": "Notes API"})
print(formatted.messages[0].content)

---

## 4. Invoking Templates — ChatPromptValue

`invoke()` returns a **`ChatPromptValue`** — a Runnable output wrapping the formatted **`messages`** list.

In [ ]:
prompt_value = multi_role.invoke({
    "topic": "JWT authentication",
    "audience": "junior developer",
    "question": "What header carries the token?",
})

print("type:", type(prompt_value).__name__)
print("to_string():\n", prompt_value.to_string())
print()
for m in prompt_value.messages:
    print(f"  [{m.type}] {m.content}")

### 4.1 format_messages vs invoke

| Method | Returns |
|--------|--------|
| **`invoke(dict)`** | `ChatPromptValue` (Runnable standard) |
| **`format_messages(dict)`** | `list[BaseMessage]` directly |

Use `format_messages` when you need messages without going through the Runnable API; use `invoke` inside LCEL chains.

In [ ]:
msgs = multi_role.format_messages({
    "topic": "Alembic",
    "audience": "backend developer",
    "question": "What command applies migrations?",
})
print([m.type for m in msgs])

---

## 5. Role Tuples — System, Human, AI

`from_messages` accepts `(role, content)` tuples. Standard roles:

| Tuple role | Message class |
|------------|---------------|
| `"system"` | `SystemMessage` |
| `"human"` or `"user"` | `HumanMessage` |
| `"ai"` or `"assistant"` | `AIMessage` |

Separating roles helps models follow instructions: system rules persist across turns; human holds the user query.

In [ ]:
role_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}. Never exceed {word_limit} words."),
    ("human", "{user_input}"),
])

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
chain = role_prompt | llm | StrOutputParser()

print(chain.invoke({
    "persona": "strict technical reviewer",
    "word_limit": "15",
    "user_input": "What is idempotency in REST?",
}))

### 5.1 Fixed AI Examples in the Template

Include `"ai"` tuples for static demonstration turns (not model-generated — part of the template):

In [ ]:
with_example = ChatPromptTemplate.from_messages([
    ("system", "Classify sentiment as positive or negative."),
    ("human", "I love this API"),
    ("ai", "positive"),
    ("human", "{text}"),
])

pv = with_example.invoke({"text": "The deployment failed again"})
for m in pv.messages:
    print(f"[{m.type}] {m.content}")

---

## 6. Message-Level Template Classes

For fine-grained control, use explicit message prompt template classes instead of tuples:

In [ ]:
from langchain_core.prompts import (
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

explicit_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a {role} helping with {domain}."
    ),
    HumanMessagePromptTemplate.from_template("Question: {question}"),
    AIMessagePromptTemplate.from_template("Acknowledged. I will answer briefly."),
    HumanMessagePromptTemplate.from_template("Now answer: {question}"),
])

print(explicit_prompt.invoke({
    "role": "mentor",
    "domain": "FastAPI",
    "question": "What is dependency injection?",
}).to_string())

---

## 7. MessagesPlaceholder — Dynamic Message Slots

**`MessagesPlaceholder(variable_name)`** inserts a **list of messages** at runtime — essential for chat history, agent scratchpads, and multi-turn RAG.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

filled = chat_prompt.invoke({
    "history": [
        HumanMessage(content="What is Alembic?"),
        AIMessage(content="Alembic manages database schema migrations for SQLAlchemy."),
    ],
    "input": "What command upgrades the schema?",
})

print(f"{len(filled.messages)} messages in prompt:")
for m in filled.messages:
    print(f"  [{m.type}] {m.content[:60]}..." if len(m.content) > 60 else f"  [{m.type}] {m.content}")

### 7.1 optional=True — Empty History

When history may be absent, mark the placeholder optional:

In [ ]:
optional_history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful."),
    MessagesPlaceholder("history", optional=True),
    ("human", "{input}"),
])

first_turn = optional_history_prompt.invoke({"input": "Hello!"})
print("first turn message count:", len(first_turn.messages))
print([m.type for m in first_turn.messages])

Persistent session wiring with **`RunnableWithMessageHistory`** is covered in **14. Memory and Chat History**.

---

## 8. partial() — Pre-Fill Variables

**.partial()** freezes template variables that do not change per request. Remaining variables are still required at `invoke()` time.

In [ ]:
base_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}. Language: {language}. Tone: {tone}."),
    ("human", "{question}"),
])

support_bot_prompt = base_prompt.partial(
    persona="customer support agent for a Notes API",
    language="English",
    tone="friendly and professional",
)

print("remaining variables:", support_bot_prompt.input_variables)

support_chain = support_bot_prompt | llm | StrOutputParser()
print(support_chain.invoke({"question": "How do I reset my API key?"}))

### 8.1 partial() for Format Instructions

Common pattern with output parsers — inject parser format instructions once:

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


class TagResult(BaseModel):
    tags: list[str] = Field(description="Up to 3 tags")
    summary: str = Field(description="One sentence summary")


parser = JsonOutputParser(pydantic_object=TagResult)

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured data from the user text.\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

print("variables after partial:", extract_prompt.input_variables)
print("\nformat_instructions preview:")
print(parser.get_format_instructions()[:200], "...")

---

## 9. Few-Shot Prompt Templates

**Few-shot prompting** embeds input/output examples in the prompt so the model learns the pattern by demonstration. **`FewShotChatMessagePromptTemplate`** generates example message pairs from a list of dicts.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "The API is fast and reliable", "output": "positive"},
    {"input": "Documentation is confusing", "output": "negative"},
    {"input": "It works as expected", "output": "neutral"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Classify sentiment: {input}"),
    ("ai", "{output}"),
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify sentiment as positive, negative, or neutral."),
    few_shot,
    ("human", "Classify sentiment: {input}"),
])

sentiment_chain = sentiment_prompt | llm | StrOutputParser()
print(sentiment_chain.invoke({"input": "Best developer experience ever"}))

### 9.1 Different Examples per Request

Build the few-shot block with a helper when examples change per request (tenant, language, etc.):

In [ ]:
def build_sentiment_prompt(examples: list[dict]) -> ChatPromptTemplate:
    few_shot_block = FewShotChatMessagePromptTemplate(
        example_prompt=example_prompt,
        examples=examples,
    )
    return ChatPromptTemplate.from_messages([
        ("system", "Classify sentiment."),
        few_shot_block,
        ("human", "Classify sentiment: {input}"),
    ])


english_examples = [{"input": "Great!", "output": "positive"}]
prompt_en = build_sentiment_prompt(english_examples)

pv = prompt_en.invoke({"input": "Terrible latency"})
print(pv.to_string())
print("message count:", len(pv.messages))

---

## 10. Jinja2 Template Format

By default, templates use **f-string** formatting (`{var}`). For conditionals and loops, set **`template_format="jinja2"`**:

In [ ]:
jinja_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a {{ role }}.{% if strict %} Be extremely concise.{% endif %}",
        ),
        (
            "human",
            "Topics: {% for t in topics %}{{ t }}{% if not loop.last %}, {% endif %}{% endfor %}\nQuestion: {{ question }}",
        ),
    ],
    template_format="jinja2",
)

jinja_value = jinja_prompt.invoke({
    "role": "API tutor",
    "strict": True,
    "topics": ["REST", "JWT", "pytest"],
    "question": "Which topic covers auth?",
})

for m in jinja_value.messages:
    print(f"[{m.type}]\n{m.content}\n")

| Format | Syntax | Use when |
|--------|--------|----------|
| **`f-string`** (default) | `{variable}` | Simple substitution |
| **`jinja2`** | `{{ variable }}`, `{% if %}`, `{% for %}` | Conditional sections, loops |

---

## 11. Legacy PromptTemplate (String Prompts)

**`PromptTemplate`** formats a **single string** — the legacy interface for completion models. With chat models, prefer **`ChatPromptTemplate`**.

Still useful for embedding one blob into a chat template or non-LLM steps:

In [ ]:
from langchain_core.prompts import PromptTemplate

string_template = PromptTemplate.from_template(
    "Context block:\n{context}\n\n---\nEnd of context."
)

context_block = string_template.invoke({"context": "JWT uses Bearer tokens."})
print(context_block.text)

# Embed formatted string into a chat prompt
rag_style = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context."),
    ("human", string_template.template),
    ("human", "Question: {question}"),
])

print("\nrag_style variables:", rag_style.input_variables)

---

## 12. RAG Prompt Patterns (Preview)

Document Q&A prompts separate **instructions**, **context**, and **question**. Full RAG chain assembly is in **11. RAG with LCEL**.

In [ ]:
RAG_SYSTEM = """You answer questions using ONLY the context below.
If the answer is not in the context, say "I don't know."
Cite chunk ids in square brackets when possible.
"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM + "\n\nContext:\n{context}"),
    ("human", "{question}"),
])

sample = rag_prompt.invoke({
    "context": "[c3] JWT bearer tokens use Authorization: Bearer header.",
    "question": "What header carries the JWT?",
})

print(sample.messages[0].content[:120], "...")
print("---")
print(sample.messages[1].content)

### 12.1 Condense Question Template (Multi-Turn RAG Preview)

Rewrite follow-ups as standalone queries before retrieval:

In [ ]:
condense_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"),
    (
        "human",
        "Given the conversation above, rewrite this follow-up as a standalone question.\n"
        "Follow-up: {question}\n"
        "Standalone question:",
    ),
])

condense_chain = condense_prompt | llm | StrOutputParser()

standalone = condense_chain.invoke({
    "history": [
        HumanMessage(content="Tell me about Alembic migrations."),
        AIMessage(content="Alembic applies SQLAlchemy schema migrations."),
    ],
    "question": "What command do I run?",
})
print("standalone:", standalone)

---

## 13. Saving and Loading Prompts

Serialize templates for version control and deployment using **`dumpd`** / **`loads`** from `langchain_core.load`:

In [ ]:
from langchain_core.load import dumpd, loads

serialized = dumpd(multi_role)
json_blob = json.dumps(serialized, indent=2)
print("serialized keys:", list(serialized.keys())[:5], "...")

restored = loads(json_blob)
print("restored type:", type(restored).__name__)
print("restored variables:", restored.input_variables)

Store serialized prompt JSON in git; deserialize at application startup. Pair with eval suites in **17. Production Patterns for LangChain** to catch regressions when prompts change.

---

## 14. Prompt Templates in LCEL — Type Flow

```
dict ──► ChatPromptTemplate ──► ChatPromptValue ──► ChatOpenAI ──► AIMessage ──► Parser ──► str
```

The prompt step transforms a **variable dict** into **messages**. Everything before the model is deterministic string formatting — ideal for unit tests without API calls.

In [ ]:
from langchain_core.runnables import RunnableLambda


def inspect_prompt_step(inputs: dict) -> dict:
    pv = multi_role.invoke(inputs)
    return {
        "message_count": len(pv.messages),
        "roles": [m.type for m in pv.messages],
        "preview": pv.messages[-1].content,
    }


inspect_chain = RunnableLambda(inspect_prompt_step)
print(json.dumps(inspect_chain.invoke({
    "topic": "pytest",
    "audience": "developer",
    "question": "What is conftest.py?",
}), indent=2))

---

## 15. Production Prompt Guidelines

| Practice | Rationale |
|----------|----------|
| **One template per task** | Avoid mega-prompts with unrelated modes |
| **System vs human separation** | Instructions in system; user data in human |
| **`.partial()` for static config** | Persona, language, format instructions |
| **Version in git** | `.save()` / JSON or inline in repo |
| **Test with `format_messages`** | No API key needed to validate shape |
| **Cap example count in few-shot** | Token cost grows linearly with examples |
| **Sanitize user input placement** | User text in `{human}` slots, not system rules |

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Missing variable at invoke | `KeyError` | Check `input_variables` |
| User data in system message | Prompt injection risk | Move user input to human role |
| Forgetting `MessagesPlaceholder` key | Empty history slot | Pass `history` list in invoke dict |
| Single string for multi-role prompt | Model ignores instructions | Use `from_messages` |
| Too many few-shot examples | High cost, context overflow | Limit examples; use dynamic selection |
| Mixing f-string and jinja2 syntax | Format errors | Set `template_format` consistently |
| Not using `.partial()` for static text | Repeat same kwargs every call | Freeze with `.partial()` |

---

## 17. Summary

**Prompt templates** turn variable dicts into formatted **message lists** for chat models. **`ChatPromptTemplate`** is the primary class — built with **`from_messages`** or **`from_template`**, composed in LCEL, and extended with **`MessagesPlaceholder`**, **`.partial()`**, and **`FewShotChatMessagePromptTemplate`**.

Key takeaways:

- Templates are **Runnables** — `invoke()` returns **`ChatPromptValue`** with `.messages`.
- Use **role tuples** (`system`, `human`, `ai`) for clear instruction boundaries.
- **`MessagesPlaceholder`** injects dynamic message lists (history, scratchpad).
- **`.partial()`** pre-fills static variables; pair with parser format instructions in **05**.
- **Few-shot** templates embed demonstration turns before the real query.
- **Jinja2** format enables conditionals and loops in template text.

Demonstrations covered variable detection, role prompts, placeholders, partial prompts, few-shot classification, Jinja2 templates, RAG/condense patterns, save/load, and prompt inspection without API calls.

Next: **05. Output Parsers and Structured Output** — `StrOutputParser`, JSON parsers, Pydantic schemas, and `with_structured_output`.